# Прокси-модель давления в пласте (Simplified FNO) - версия 7

Базовая архитектура и данные — те же, что v5 (3 канала, 2000 сэмплов из `data/`).
Ключевое отличие: **добавлен physics-informed residual в функцию потерь** —
штраф за нарушение уравнения Лапласа ∇²p = 0 в области вдали от скважин.

Гипотеза для проверки: физическое ограничение помогает не столько поднять
средний skill, сколько **устранить катастрофические выбросы** (те самые -15
на 4 скважинах и -0.7 на 6, которые есть у v5).

Отдельно сравниваем v5 и v7 на подмножестве "трудных" сэмплов — топ-20%
худших по v5 — чтобы прицельно проверить, помогает ли физика именно там.

Запускать ячейки строго по порядку сверху вниз.

## 1. Подключение Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Импорты и проверка GPU

In [ ]:
import os, glob, time
from collections import defaultdict
import numpy as np
import scipy.io as sio
from scipy.ndimage import gaussian_filter, binary_dilation
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Пути

In [ ]:
DATA_DIR   = '/content/drive/MyDrive/proxy_model/data/'
BLIND_DIR  = '/content/drive/MyDrive/proxy_model/data_blind/'
OUT_DIR    = '/content/drive/MyDrive/proxy_model/'
MODEL_PATH_V5 = os.path.join(OUT_DIR, 'best_model_v5.pt')   # для сравнения
MODEL_PATH    = os.path.join(OUT_DIR, 'best_model_v7.pt')

files = sorted(glob.glob(os.path.join(DATA_DIR, 'sample_*.mat')))
blind_files = sorted(glob.glob(os.path.join(BLIND_DIR, 'blind_*.mat')))

print('train/val файлов:', len(files))
print('слепых файлов   :', len(blind_files))
assert len(files) > 0, 'Файлы не найдены - проверь DATA_DIR'

## 4. Чтение данных и сборка каналов (как в v5)

In [ ]:
WELL_SIGMA = 1.5

_ref = np.zeros((15, 15)); _ref[7, 7] = 1.0
_PEAK = gaussian_filter(_ref, sigma=WELL_SIGMA, mode='constant').max()


def smooth_mask(points, nx, ny):
    m = np.zeros((nx, ny))
    for (i, j) in points:
        m[i, j] += 1.0
    m = gaussian_filter(m, sigma=WELL_SIGMA, mode='constant')
    return m / _PEAK


def well_exclusion_mask(inj_pts, prod_pts, nx, ny, radius=3):
    """Бинарная маска: True там, где физика Лапласа применима (вдали от скважин).
    Скважины и radius клеток вокруг них исключаются."""
    m = np.zeros((nx, ny), dtype=bool)
    for (i, j) in inj_pts + prod_pts:
        m[i, j] = True
    m = binary_dilation(m, iterations=radius)
    return ~m   # инвертируем: True = "здесь физика должна выполняться"


def build_sample(d):
    perm = np.asarray(d['perm_field'], dtype=np.float64)
    pres = np.asarray(d['pressure_field'], dtype=np.float64) if 'pressure_field' in d else None

    coords = np.asarray(d['well_coords']).astype(int)
    wtype  = np.asarray(d['well_type']).astype(int).ravel()

    nx, ny = perm.shape
    inj_pts  = [(I - 1, J - 1) for (I, J), t in zip(coords, wtype) if t == 1]
    prod_pts = [(I - 1, J - 1) for (I, J), t in zip(coords, wtype) if t != 1]

    return {
        'perm':  perm,
        'inj':   smooth_mask(inj_pts,  nx, ny),
        'prod':  smooth_mask(prod_pts, nx, ny),
        'phys_mask': well_exclusion_mask(inj_pts, prod_pts, nx, ny, radius=3),
        'pres':  pres,
        'inj_pts':  inj_pts,
        'prod_pts': prod_pts,
    }


perm_all, inj_all, prod_all, pres_all, phys_mask_all = [], [], [], [], []
pts_all = []

t0 = time.time()
for k, fpath in enumerate(files):
    s = build_sample(sio.loadmat(fpath))
    perm_all.append(s['perm'])
    inj_all.append(s['inj'])
    prod_all.append(s['prod'])
    pres_all.append(s['pres'])
    phys_mask_all.append(s['phys_mask'])
    pts_all.append((s['inj_pts'], s['prod_pts']))
    if (k + 1) % 200 == 0:
        print(f'прочитано {k+1}/{len(files)}')

perm_all      = np.stack(perm_all)
inj_all       = np.stack(inj_all)
prod_all      = np.stack(prod_all)
pres_all      = np.stack(pres_all)
phys_mask_all = np.stack(phys_mask_all).astype(np.float32)

print(f'\nготово за {time.time()-t0:.1f} c')
print('perm      :', perm_all.shape)
print('phys_mask :', phys_mask_all.shape,
      f'доля клеток с физикой: {phys_mask_all.mean():.2%}')

## 5. Нормализация и train/val split

In [ ]:
log_perm = np.log10(perm_all)

NORM = {
    'KMIN': float(log_perm.min()),  'KMAX': float(log_perm.max()),
    'PMIN': float(pres_all.min()),  'PMAX': float(pres_all.max()),
    'WELL_SIGMA': WELL_SIGMA,
}

perm_n = (log_perm - NORM['KMIN']) / (NORM['KMAX'] - NORM['KMIN'])
pres_n = (pres_all - NORM['PMIN']) / (NORM['PMAX'] - NORM['PMIN'])

print(f"log10(k): [{NORM['KMIN']:.3f}, {NORM['KMAX']:.3f}]")
print(f"давление: [{NORM['PMIN']:.2f}, {NORM['PMAX']:.2f}] бар")

X = np.stack([perm_n, inj_all, prod_all], axis=-1)   # (N, 40, 40, 3)
Y = pres_n
M = phys_mask_all                                     # физ. маска для loss
print('X:', X.shape, ' Y:', Y.shape, ' M:', M.shape)

N = X.shape[0]
NVAL   = max(1, int(round(0.2 * N)))
NTRAIN = N - NVAL

rng = np.random.RandomState(SEED)
idx = rng.permutation(N)
tr_idx, va_idx = idx[:NTRAIN], idx[NTRAIN:]

X_tr, Y_tr, M_tr = X[tr_idx], Y[tr_idx], M[tr_idx]
X_va, Y_va, M_va = X[va_idx], Y[va_idx], M[va_idx]
pts_va = [pts_all[k] for k in va_idx]

print('train:', X_tr.shape[0], ' val:', X_va.shape[0])

## 6. Аугментация отражениями

Отражаем и физ. маску вместе со всем остальным — она согласована с положением скважин, а те тоже отражаются.

In [ ]:
AUGMENT = True

if AUGMENT:
    def flip_set(Xa, Ya, Ma):
        Xs = [Xa, np.flip(Xa, axis=1), np.flip(Xa, axis=2),
              np.flip(np.flip(Xa, axis=2), axis=1)]
        Ys = [Ya, np.flip(Ya, axis=1), np.flip(Ya, axis=2),
              np.flip(np.flip(Ya, axis=2), axis=1)]
        Ms = [Ma, np.flip(Ma, axis=1), np.flip(Ma, axis=2),
              np.flip(np.flip(Ma, axis=2), axis=1)]
        return (np.concatenate(Xs, 0).copy(),
                np.concatenate(Ys, 0).copy(),
                np.concatenate(Ms, 0).copy())

    X_tr, Y_tr, M_tr = flip_set(X_tr, Y_tr, M_tr)
    print('train после аугментации:', X_tr.shape[0])

X_tr_t = torch.from_numpy(X_tr).float()
Y_tr_t = torch.from_numpy(Y_tr).float()
M_tr_t = torch.from_numpy(M_tr).float()
X_va_t = torch.from_numpy(X_va).float()
Y_va_t = torch.from_numpy(Y_va).float()
M_va_t = torch.from_numpy(M_va).float()

BATCH = 10
train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_tr_t, Y_tr_t, M_tr_t),
    batch_size=BATCH, shuffle=True)
val_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_va_t, Y_va_t, M_va_t),
    batch_size=BATCH, shuffle=False)

## 7. Модель (архитектура точно как в v5)

In [ ]:
IN_CH  = 3
MODES  = 18
WIDTH  = 32
LAYERS = 3


class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super().__init__()
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2

        scale = 1.0 / (in_channels * out_channels)
        self.weights1 = nn.Parameter(
            scale * torch.rand(in_channels, out_channels, modes1, modes2, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(
            scale * torch.rand(in_channels, out_channels, modes1, modes2, dtype=torch.cfloat))

    @staticmethod
    def compl_mul2d(inp, weights):
        return torch.einsum('bixy,ioxy->boxy', inp, weights)

    def forward(self, x):
        b, _, nx, ny = x.shape
        x_ft = torch.fft.rfft2(x)
        out_ft = torch.zeros(b, self.out_channels, nx, ny // 2 + 1,
                             dtype=torch.cfloat, device=x.device)
        out_ft[:, :, :self.modes1, :self.modes2] = self.compl_mul2d(
            x_ft[:, :, :self.modes1, :self.modes2], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2] = self.compl_mul2d(
            x_ft[:, :, -self.modes1:, :self.modes2], self.weights2)
        return torch.fft.irfft2(out_ft, s=(nx, ny))


class SimpleFNO(nn.Module):
    def __init__(self, in_channels=IN_CH, modes1=MODES, modes2=MODES,
                 width=WIDTH, n_layers=LAYERS):
        super().__init__()
        self.width = width
        self.fc0 = nn.Linear(in_channels, width)
        self.specs = nn.ModuleList(
            [SpectralConv2d(width, width, modes1, modes2) for _ in range(n_layers)])
        self.ws = nn.ModuleList(
            [nn.Conv1d(width, width, 1) for _ in range(n_layers)])
        self.fc1 = nn.Linear(width, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        b, nx, ny, _ = x.shape
        x = self.fc0(x)
        x = x.permute(0, 3, 1, 2)
        for spec, w in zip(self.specs, self.ws):
            x1 = spec(x)
            x2 = w(x.reshape(b, self.width, -1)).reshape(b, self.width, nx, ny)
            x = F.relu(x1 + x2)
        x = x.permute(0, 2, 3, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x.squeeze(-1)


model = SimpleFNO().to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'параметров в модели: {n_params:,}')

## 8. Функции потерь: data loss + physics residual

**Data loss** — та же relative L2, что в v5.

**Physics residual** — конечно-разностный лапласиан:
∇²p[i,j] ≈ p[i+1,j] + p[i-1,j] + p[i,j+1] + p[i,j-1] - 4·p[i,j]

Штраф — среднеквадратичное значение лапласиана, взятое **только по клеткам,
где физ. маска = 1** (то есть вдали от скважин).

Выбираем конечные разности, а не autograd по координатам — быстрее и проще
на дискретной сетке фиксированного размера.

In [ ]:
class LpLoss(object):
    def __init__(self, p=2):
        self.p = p
    def __call__(self, x, y):
        n = x.size(0)
        diff = torch.norm(x.reshape(n, -1) - y.reshape(n, -1), self.p, 1)
        base = torch.norm(y.reshape(n, -1), self.p, 1)
        return torch.mean(diff / base)


def laplacian_2d(p):
    """Конечно-разностный лапласиан на 2D сетке. p: (b, nx, ny). Возвращает (b, nx-2, ny-2)."""
    return (p[:, 2:,   1:-1] +
            p[:, :-2,  1:-1] +
            p[:, 1:-1, 2:]   +
            p[:, 1:-1, :-2]  -
            4 * p[:, 1:-1, 1:-1])


def physics_residual(pred, phys_mask):
    """Средний квадрат лапласиана только в клетках, где phys_mask=1.
    Обрезаем маску так же, как и лапласиан (по 1 клетке с каждой стороны)."""
    lap = laplacian_2d(pred)
    m   = phys_mask[:, 1:-1, 1:-1]
    lap_sq = (lap ** 2) * m
    denom = m.sum(dim=(1, 2)).clamp(min=1.0)
    return (lap_sq.sum(dim=(1, 2)) / denom).mean()


data_loss = LpLoss()

## 9. Обучение с комбинированным loss

`total = data_loss + LAMBDA_PHYS * physics_residual`

Начнём с фиксированного веса `LAMBDA_PHYS = 0.01` — подобран так, чтобы физический
член был примерно на порядок меньше data loss в начале обучения. Если результат
окажется нейтральным или хуже — попробуем другие веса или адаптивное расписание.

Отдельно логируем два компонента loss раздельно, чтобы видеть, как они соотносятся
в динамике.

In [ ]:
EPOCHS = 300
LR = 1e-3
STEP = 50
LAMBDA_PHYS = 0.01     # вес физического члена в общем loss

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP, gamma=0.5)

os.makedirs(OUT_DIR, exist_ok=True)
PSPAN = NORM['PMAX'] - NORM['PMIN']

train_hist, val_hist, mae_hist, skill_hist = [], [], [], []
data_hist, phys_hist = [], []
best_val = float('inf')
t0 = time.time()

for ep in range(EPOCHS):
    model.train()
    tr_loss, tr_data, tr_phys = 0.0, 0.0, 0.0
    for xb, yb, mb in train_loader:
        xb, yb, mb = xb.to(device), yb.to(device), mb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        L_data = data_loss(pred, yb)
        L_phys = physics_residual(pred, mb)
        loss = L_data + LAMBDA_PHYS * L_phys
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * xb.size(0)
        tr_data += L_data.item() * xb.size(0)
        tr_phys += L_phys.item() * xb.size(0)
    tr_loss /= len(train_loader.dataset)
    tr_data /= len(train_loader.dataset)
    tr_phys /= len(train_loader.dataset)

    model.eval()
    va_loss, mae_bar = 0.0, 0.0
    sse_model, sse_mean = 0.0, 0.0
    with torch.no_grad():
        for xb, yb, mb in val_loader:
            xb, yb, mb = xb.to(device), yb.to(device), mb.to(device)
            pred = model(xb)
            va_loss += data_loss(pred, yb).item() * xb.size(0)
            mae_bar += (torch.abs(pred - yb).mean() * PSPAN).item() * xb.size(0)
            base = yb.mean(dim=(1, 2), keepdim=True)
            sse_model += ((pred - yb) ** 2).sum().item()
            sse_mean  += ((base - yb) ** 2).sum().item()
    va_loss /= len(val_loader.dataset)
    mae_bar /= len(val_loader.dataset)
    skill = 1.0 - sse_model / sse_mean

    scheduler.step()
    train_hist.append(tr_loss); val_hist.append(va_loss)
    mae_hist.append(mae_bar);   skill_hist.append(skill)
    data_hist.append(tr_data);  phys_hist.append(tr_phys)

    if va_loss < best_val:
        best_val = va_loss
        torch.save({'model_state': model.state_dict(),
                    'norm': NORM,
                    'arch': {'in_channels': IN_CH, 'modes': MODES,
                             'width': WIDTH, 'n_layers': LAYERS},
                    'lambda_phys': LAMBDA_PHYS,
                    'epoch': ep, 'val_loss': float(va_loss)}, MODEL_PATH)

    if ep % 10 == 0 or ep == EPOCHS - 1:
        print(f'ep {ep:3d} | data {tr_data:.4f} | phys {tr_phys:.4f} '
              f'| val {va_loss:.4f} | MAE {mae_bar:.2f} б | skill {skill:.3f}')

print(f'\nобучение заняло {(time.time()-t0)/60:.1f} мин')
print(f'лучший val loss (data-only): {best_val:.4f}')
print(f'финальный skill: {skill_hist[-1]:.3f}')

## 10. Кривые обучения (с раздельными компонентами loss)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

axes[0].plot(data_hist, label='data loss (train)', color='tab:blue')
axes[0].plot(val_hist,  label='data loss (val)',   color='tab:orange')
axes[0].set_xlabel('эпоха'); axes[0].set_ylabel('rel. L2')
axes[0].set_yscale('log'); axes[0].legend(); axes[0].grid(alpha=.3)
axes[0].set_title('Data loss')

axes[1].plot(phys_hist, color='tab:green')
axes[1].set_xlabel('эпоха'); axes[1].set_ylabel('средний ||∇²p||²')
axes[1].set_yscale('log'); axes[1].grid(alpha=.3)
axes[1].set_title('Physics residual (train)')

axes[2].plot(skill_hist, color='tab:red')
axes[2].axhline(0, color='k', lw=.8, ls='--')
axes[2].set_xlabel('эпоха'); axes[2].set_ylabel('skill')
axes[2].set_ylim(-0.1, 1.0); axes[2].grid(alpha=.3)
axes[2].set_title('Skill (val)')

plt.tight_layout(); plt.show()

## 11. Blind-test v7 по группам

In [ ]:
ck = torch.load(MODEL_PATH, map_location=device, weights_only=False)
nm, ar = ck['norm'], ck['arch']

net = SimpleFNO(in_channels=ar['in_channels'], modes1=ar['modes'],
                modes2=ar['modes'], width=ar['width'],
                n_layers=ar['n_layers']).to(device)
net.load_state_dict(ck['model_state'])
net.eval()


def predict_and_score(fpath, network, nm):
    s = build_sample(sio.loadmat(fpath))
    perm_n = (np.log10(s['perm']) - nm['KMIN']) / (nm['KMAX'] - nm['KMIN'])
    x = np.stack([perm_n, s['inj'], s['prod']], axis=-1)[None, ...]

    with torch.no_grad():
        pred = network(torch.from_numpy(x).float().to(device)).cpu().numpy()[0]
    pred_bar = pred * (nm['PMAX'] - nm['PMIN']) + nm['PMIN']
    true_bar = s['pres']

    mae = np.abs(pred_bar - true_bar).mean()
    rel = np.linalg.norm(pred_bar - true_bar) / np.linalg.norm(true_bar)
    sk  = 1 - ((pred_bar - true_bar) ** 2).sum() / ((true_bar.mean() - true_bar) ** 2).sum()
    nw = len(s['inj_pts']) + len(s['prod_pts'])
    return mae, rel, sk, nw


res_v7 = defaultdict(list)
per_file_v7 = {}   # {basename: (mae, rel, skill)}

for fpath in blind_files:
    mae, rel, sk, nw = predict_and_score(fpath, net, nm)
    res_v7[nw].append((mae, rel, sk))
    per_file_v7[os.path.basename(fpath)] = (mae, rel, sk)

print(f'{"скважин":>8} | {"N":>4} | {"MAE, бар":>9} | {"отн.L2":>7} | '
      f'{"skill сред":>10} | {"skill мед":>9} | {"skill худш":>10}')
print('-' * 78)
for nw in sorted(res_v7):
    a = np.array(res_v7[nw])
    print(f'{nw:>8} | {len(a):>4} | {a[:,0].mean():>9.2f} | {a[:,1].mean()*100:>6.2f}% | '
          f'{a[:,2].mean():>10.3f} | {np.median(a[:,2]):>9.3f} | {a[:,2].min():>10.3f}')

## 12. Прицельное сравнение v5 vs v7 на "трудных" случаях

Логика: загружаем v5, оцениваем каждый файл слепой выборки, находим топ-20% худших
по skill. Именно на них смотрим, помог ли physics residual — если да, здесь скачок
skill будет заметнее всего.

In [ ]:
assert os.path.exists(MODEL_PATH_V5), 'best_model_v5.pt не найден - запусти v5 сначала'

ck5 = torch.load(MODEL_PATH_V5, map_location=device, weights_only=False)
nm5, ar5 = ck5['norm'], ck5['arch']

net5 = SimpleFNO(in_channels=ar5['in_channels'], modes1=ar5['modes'],
                 modes2=ar5['modes'], width=ar5['width'],
                 n_layers=ar5['n_layers']).to(device)
net5.load_state_dict(ck5['model_state'])
net5.eval()

# --- прогоняем v5 по всем слепым файлам ---
per_file_v5 = {}
for fpath in blind_files:
    mae, rel, sk, nw = predict_and_score(fpath, net5, nm5)
    per_file_v5[os.path.basename(fpath)] = (mae, rel, sk, nw)

# --- топ-20% худших по v5 ---
sorted_by_v5 = sorted(per_file_v5.items(), key=lambda kv: kv[1][2])   # по skill (возр.)
n_hard = max(1, len(sorted_by_v5) // 5)
hard_names = [name for name, _ in sorted_by_v5[:n_hard]]

print(f'"Трудные" сэмплы по v5 (топ {n_hard} худших по skill):')
print(f'  порог skill v5 <= {sorted_by_v5[n_hard-1][1][2]:.3f}')
print()

# --- сравнение ---
diff_skills = []
print(f'{"файл":>32} | {"nw":>2} | {"skill v5":>9} | {"skill v7":>9} | {"Δ":>+7}')
print('-' * 75)
for name in hard_names:
    sk_v5 = per_file_v5[name][2]
    sk_v7 = per_file_v7[name][2]
    nw    = per_file_v5[name][3]
    d = sk_v7 - sk_v5
    diff_skills.append(d)
    print(f'{name:>32} | {nw:>2} | {sk_v5:>9.3f} | {sk_v7:>9.3f} | {d:+7.3f}')

diffs = np.array(diff_skills)
print()
print(f'Средний Δskill на трудных случаях: {diffs.mean():+.3f}')
print(f'Улучшилось: {(diffs > 0).sum()} из {len(diffs)}')
print(f'Ухудшилось: {(diffs < 0).sum()} из {len(diffs)}')

# на всех слепых для контекста
all_v5 = np.array([v[2] for v in per_file_v5.values()])
all_v7 = np.array([per_file_v7[n][2] for n in per_file_v5.keys()])
print()
print(f'Для сравнения: средний Δskill на ВСЕЙ слепой выборке: '
      f'{(all_v7 - all_v5).mean():+.3f}')

## 13. Визуализация: boxplot сравнение v5 и v7

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- слева: по группам ---
positions = []
data_box = []
labels = []
for i, nw in enumerate(sorted(res_v7)):
    # v5 по этой же группе
    v5_group = [per_file_v5[n][2] for n in per_file_v5
                if per_file_v5[n][3] == nw]
    v7_group = [per_file_v7[n][2] for n in per_file_v7
                if per_file_v5[n][3] == nw]

    positions.extend([i * 3, i * 3 + 1])
    data_box.extend([v5_group, v7_group])
    labels.extend([f'v5\n{nw} скв.', f'v7\n{nw} скв.'])

bp = axes[0].boxplot(data_box, positions=positions, widths=0.8, patch_artist=True,
                     tick_labels=labels)
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor('lightsteelblue' if i % 2 == 0 else 'lightsalmon')
axes[0].axhline(0, color='r', lw=.9, ls='--')
axes[0].set_ylabel('skill')
axes[0].set_title('Skill по группам скважин: v5 vs v7')
axes[0].grid(alpha=.3, axis='y')

# --- справа: только на трудных ---
sk_v5_hard = [per_file_v5[n][2] for n in hard_names]
sk_v7_hard = [per_file_v7[n][2] for n in hard_names]

axes[1].scatter(sk_v5_hard, sk_v7_hard, alpha=.6, s=50)
low = min(min(sk_v5_hard), min(sk_v7_hard)) - 0.5
axes[1].plot([low, 1], [low, 1], 'r--', lw=.9, label='v7 = v5')
axes[1].axhline(0, color='k', lw=.5, ls=':')
axes[1].axvline(0, color='k', lw=.5, ls=':')
axes[1].set_xlabel('skill v5'); axes[1].set_ylabel('skill v7')
axes[1].set_title(f'Скаттер на {len(hard_names)} трудных случаях (выше линии = улучшение)')
axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout(); plt.show()

## 14. Итоги эксперимента

Заполни после запуска:

| Метрика | v5 (без физики) | v7 (с физикой) | Δ |
|---|---|---|---|
| val loss (data) | 0.064 | ? | |
| skill сред. (все 100 blind) | | | |
| skill мед. (все 100) | | | |
| skill сред. (топ-20% трудных) | | | |
| Число улучшившихся из 20 трудных | — | ? | |

**Возможные исходы и их интерпретация:**
- Δskill > 0 на трудных, ≈ 0 на всех: физика помогает именно в проблемных зонах, для продакшена стоит включать
- Δskill ≈ 0 везде: физика не влияет — задача слишком простая или вес слишком мал, попробовать увеличить LAMBDA_PHYS
- Δskill < 0: физика мешает — либо жёсткое ограничение конфликтует с данными, либо маска исключения скважин слишком узкая